# Group images

Find correlated groups in dataset.

Algorithm:
1. Read names from CVS file
2. Remove duplicates
3. Group by prefix similarity
4. Calculate image embeddings and image similarities
5. Group by avg image similarities in groups
6. Save groups' ID

In [1]:
import os
import json
import uuid
import random

import numpy as np
import pandas as pd
import numpy as np
import cv2
from transformers import AutoFeatureExtractor, AutoModel
import torch
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

/home/klymy/.pyenv/versions/3.10.13/envs/pgpytrain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Read names from CVS file. Remove duplicates

In [2]:
# BASE_PATH = "/mnt/pg-fs01/Current/DB/ObjectDetectionV3"
BASE_PATH = "../../../data/ObjectDetectionV3"

df = pd.read_csv(os.path.join(BASE_PATH, "images.csv"))
print("'images' table shape", df.shape)

names = df["name"].values
names = np.unique(names)
print("number of unique images", len(names))

df.head()

'images' table shape (1457, 6)
number of unique images 1395


,id,name,width,height,batch,folder
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam3_H243_discount_14.png,640.0,480.0,batch1_10.5,batch1
1,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,Cam3_H243_discount_15.png,640.0,480.0,batch1_10.5,batch1
2,ff50bde4-3825-47b8-9cab-cc97663f1c97,Cam3_H243_discount_16.png,640.0,480.0,batch1_10.5,batch1
3,5cec4eb5-edd9-4831-9ca3-5cfb04fc6d82,Cam3_H243_discount_17.png,640.0,480.0,batch1_10.5,batch1
4,b92da22b-21df-406f-8a0b-3c3336d8393a,Cam3_H243_discount_18.png,640.0,480.0,batch1_10.5,batch1


## Group by prefix similarity

### Build prefix hierarchy

In [3]:
class Node:
    def __init__(self, prefix, count, parrent_prefix=""):
        self.prefix = prefix
        self.parrent_prefix = parrent_prefix
        self.leafs = []
        self.count = count

        self.full_name = f"{self.parrent_prefix}_{self.prefix}" if self.parrent_prefix else self.prefix

    def __str__(self):
        return f"'{self.full_name}': {self.count}"

def build_tree(filenames, prefix="", parrent_prefix=""):
    if prefix == "Cam3_H250_spaceKS":
            pass

    node = Node(prefix, len(filenames), parrent_prefix)
    leafs = []
    
    if len(filenames) == 1 and filenames[0] == "":
        return node
    
    filename_parts = [filename.split("_", 1) for filename in filenames]
    prefixes, suffixes= zip(*[i if len(i) > 1 else i + [""] for i in filename_parts])
    
    filename_parts = [filename.split("_", 1) for filename in filenames]
    prefixes, suffixes = zip(*[i if len(i) > 1 else i + [""] for i in filename_parts])

    uprefixes, inverse_idxs = np.unique(prefixes, return_inverse=True)
    
    for idx, uprefix in enumerate(uprefixes):
        leaf = build_tree(np.array(suffixes)[inverse_idxs == idx], uprefix, node.full_name)
        leafs.append(leaf)

    node.leafs = leafs
    return node


def print_tree(node, level=0, is_last=False, prefix=''):
    if level == 0:
        print(node)
    else:
        print(prefix, '└───' if is_last else '├───', node, sep='')
    prefix += '    ' if is_last else '│   '
    for i, leaf in enumerate(node.leafs):
        print_tree(leaf, level + 1, i == len(node.leafs) - 1, prefix)

In [4]:
root = build_tree(names)
print_tree(root)

'': 1395
│   ├───'2017': 16
│   │   ├───'2017_06': 13
│   │   │   └───'2017_06_21': 13
│   │   │       └───'2017_06_21_DL': 13
│   │   │           └───'2017_06_21_DL_SL': 13
│   │   │               └───'2017_06_21_DL_SL_Farnel02': 13
│   │   │                   └───'2017_06_21_DL_SL_Farnel02_H270': 13
│   │   │                       └───'2017_06_21_DL_SL_Farnel02_H270_15m': 13
│   │   │                           ├───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750': 2
│   │   │                           │   └───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750_f': 2
│   │   │                           │       ├───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750_f_00312': 1
│   │   │                           │       │   └───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750_f_00312_02122-11': 1
│   │   │                           │       │       └───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750_f_00312_02122-11_1985.jpg': 1
│   │   │                           │       └───'2017_06_21_DL_SL_Farnel

### Split hierarchy on groups

In [5]:
def get_all_leafs(node):
    if len(node.leafs) == 0:
        return [node]
    else:
        leafs = []
        for leaf in node.leafs:
            leafs += get_all_leafs(leaf)
    return leafs


def get_all_leaf_full_names(node):
    return [leaf.full_name for leaf in get_all_leafs(node)]


def is_next_level_has_the_same_count(node):
    return len(node.leafs) == 1 and node.count == node.leafs[0].count


def split_by_groups(node, groups, level=0, prev_count=-1):
    if level >= 5:
        groups[node.full_name] = get_all_leaf_full_names(node)
    elif sum([leaf.count for leaf in node.leafs]) == len(node.leafs):
        groups[node.full_name] = get_all_leaf_full_names(node)
    elif is_next_level_has_the_same_count(node):
        groups[node.full_name] = get_all_leaf_full_names(node)
    else:
        for leaf in node.leafs:
            split_by_groups(leaf, groups, level + 1, node.count)


In [6]:
groups = {}
split_by_groups(root, groups)

print("number of groups:", len(groups))
print("is all images represented in groups:", 
      sum([len(group) for group in groups.values()]) == names.shape[0])

number of groups: 185
is all images represented in groups: True


## Calculate image embeddings and image similarities

In [7]:
model_ckpt = "google/vit-base-patch16-224-in21k"
extractor = AutoFeatureExtractor.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)
hidden_dim = model.config.hidden_size


# Data transformation chain.
transformation_chain = T.Compose(
    [
        # We first resize the input image to 256x256 and then we take center crop.
        T.Resize((extractor.size["width"], extractor.size["height"])),
        T.ToTensor(),
        T.Normalize(mean=extractor.image_mean, std=extractor.image_std),
    ]
)


def extract_embeddings(model: torch.nn.Module):
    """Utility to compute embeddings."""
    device = model.device

    def pp(batch):
        images = batch["image"]
        image_batch_transformed = torch.stack(
            [transformation_chain(image) for image in images]
        )
        new_batch = {"pixel_values": image_batch_transformed.to(device)}
        with torch.no_grad():
            embeddings = model(**new_batch).last_hidden_state[:, 0].cpu()
        return {"embeddings": embeddings}

    return pp


def extract_embedding_from_img(img: np.ndarray, model: torch.nn.Module):
    """Utility to extract embeddings from a single image."""
    device = model.device
    img = transformation_chain(img)
    img = img.unsqueeze(0).to(device)
    with torch.no_grad():
        embeddings = model(pixel_values=img).last_hidden_state[:, 0].cpu()
    return embeddings


def compute_scores(emb_one, emb_two):
    """Computes cosine similarity between two vectors."""
    scores = torch.nn.functional.cosine_similarity(emb_one, emb_two)
    return scores.numpy().tolist()


def img_by_name(img_name: str, df):
    row = df[df["name"] == img_name]
    pth = os.path.join(BASE_PATH, row["batch"].iloc[0], row["folder"].iloc[0], img_name)
    img = cv2.cvtColor(cv2.imread(pth), cv2.COLOR_BGR2RGB)
    return img

/home/klymy/.pyenv/versions/3.10.13/envs/pgpytrain/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/klymy/.pyenv/versions/3.10.13/envs/pgpytrain/lib/python3.10/site-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


embeddings = []
for i in tqdm(names, desc="Extracting embeddings"):
    img = img_by_name(i, df)
    embeddings.append(extract_embedding_from_img(Image.fromarray(img), model.to(device)))


sim_scores = np.zeros((len(embeddings), len(embeddings)))
for i in tqdm(range(len(embeddings)), desc="Computing similarity scores"):
    for j in range(i, len(embeddings)):
        similarity = compute_scores(embeddings[i], embeddings[j])[0]
        sim_scores[i, j] = similarity
        sim_scores[j, i] = similarity


with open("similarity_scores.json", "w") as f:
    json.dump({"names": names.tolist(), "score_mtx": sim_scores.tolist()}, f)
with open("embeddings.json", "w") as f:
    json.dump({i: j.cpu().tolist() for i, j in zip(names, embeddings) }, f)

Device: cuda


Computing similarity scores: 100%|██████████| 1395/1395 [00:30<00:00, 46.00it/s] 


## Group by avg image similarities in groups

In [9]:
def dfs(node, visited, M, component):
    visited[node] = True
    component.append(node)
    for i, val in enumerate(M[node]):
        if val == 1 and not visited[i]:
            dfs(i, visited, M, component)

def find_connected_components(M):
    visited = [False] * len(M)
    components = []
    for i in range(len(M)):
        if not visited[i]:
            component = []
            dfs(i, visited, M, component)
            components.append(component)
    return components

# Assuming you have a 2D numpy array
M = np.array([[0, 1, 0, 0], [1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0]])

components = find_connected_components(M)
print(components)  # Outputs: [[0, 1, 2, 3]]

# Example usage:
matrix = np.array([[1, 1, 0, 0],
                   [1, 1, 0, 0],
                   [0, 0, 1, 1],
                   [0, 0, 1, 1]])

connected_components = find_connected_components(matrix)
print(connected_components)

[[0, 1, 2, 3]]
[[0, 1], [2, 3]]


In [10]:
def get_elements(x, row_indices, col_indices):
    return np.array([x[i, j] for i in row_indices for j in col_indices])


def compute_group_similarities(group_names: dict, sim_mtx: np.ndarray) -> tuple:
    avg_inner_group_similarities = []
    for i in tqdm(range(len(group_names)), desc="Computing inner group similarities"):
        indices0 = np.where(np.isin(names, groups[group_names[i]]))[0]
        avg_inner_group_similarities.append(np.mean(get_elements(sim_mtx, indices0, indices0)))

    avg_outer_goup_similarities = np.zeros((len(group_names), len(group_names)))
    max_outer_group_similarities = np.zeros((len(group_names), len(group_names)))
    for i in tqdm(range(len(group_names) - 1), desc="Computing outer group similarities"):
        for j in range(i + 1, len(group_names)):
            indices0 = np.where(np.isin(names, groups[group_names[i]]))[0]
            indices1 = np.where(np.isin(names, groups[group_names[j]]))[0]

            sims = get_elements(sim_mtx, indices0, indices1)
            avg_outer_goup_similarities[i, j] = avg_outer_goup_similarities[j, i] = np.mean(sims)
            max_outer_group_similarities[i, j] = max_outer_group_similarities[j, i] = np.max(sims)

    return avg_inner_group_similarities, avg_outer_goup_similarities, max_outer_group_similarities

In [11]:
avg_inner_group_similarities, avg_outer_goup_similarities, max_outer_group_similarities = \
    compute_group_similarities(list(groups.keys()), sim_scores)

Computing outer group similarities: 100%|██████████| 184/184 [00:07<00:00, 25.79it/s]


In [12]:
# 0.55 >= good result
# 0.6 - "golden" middle
# 65 - not so good
th = 0.6
conn_mtx = (avg_outer_goup_similarities >= th).astype(int)
connected_components = find_connected_components(conn_mtx)

print(len(connected_components))

connected_components = sorted(connected_components, key=lambda x: len(x), reverse=True)

152


### Results visualization

In [13]:
def draw_group(group_items: list, group_name: str, n: int = 3):
    if len(group_items) < n:
        subgroup = group_items
    else:
        subgroup = np.random.choice(group_items, n, replace=False)
    plt.figure(figsize=(15, 4))
    plt.suptitle(group_name, fontsize=14)
    for i, item in enumerate(subgroup):
        row = df[df["name"] == item]
        pth = os.path.join(BASE_PATH, row["batch"].iloc[0], row["folder"].iloc[0], item)
        img = cv2.cvtColor(cv2.imread(pth), cv2.COLOR_BGR2RGB)
        plt.subplot(1, len(subgroup), i + 1)
        plt.imshow(img)
        plt.title(item)

In [14]:
# for name in np.array(list(groups.keys()))[connected_components[0]]:
#     draw_group(groups[name], name)

In [15]:
# for name in np.array(list(groups.keys()))[connected_components[7]]:
#     draw_group(groups[name], name)

### Regroup images

In [16]:
rd = random.Random()
rd.seed(43)
gen_uuid = lambda: str(uuid.UUID(int=rd.getrandbits(128), version=4))

In [17]:
group_names = list(groups.keys())

joined_groups = []
for i in connected_components:
    tmp_group = []
    for j in i:
        tmp_group += groups[group_names[j]]
    joined_groups.append(tmp_group)

sum([len(i) for i in joined_groups]) == len(names)

True

In [19]:
group_ids = [gen_uuid() for _ in joined_groups]

df["group_id"] = [""] * df.shape[0]

for i, group in enumerate(joined_groups):
    df.loc[df["name"].isin(group), "group_id"] = group_ids[i]

print(df["group_id"].nunique() == len(group_ids))

df.to_csv(os.path.join(BASE_PATH, "images_grouped.csv"), index=False)

True
